In [1]:
import os
import torch
from torch.nn.functional import cross_entropy
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_from_disk, load_dataset
from tqdm import tqdm

In [2]:
from display_token_ppls import display_ppl_in_notebook

In [3]:
main_device = 'auto'
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [4]:
torch.random.manual_seed(19260817)
TOTAL_ENTRIES = 1

In [5]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-30B-A3B", device_map=main_device, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2",)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B", device_map=main_device)

dataset = load_dataset("lmsys/lmsys-chat-1m", split="train")

Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

In [6]:
model.eval()
model.config.use_cache = False
model.config.output_logits = True

In [7]:
def calc_perplexity(logits, token_ids):
  assert logits.shape[:-1] == token_ids.shape, \
      f"Logits shape {logits.shape} does not match token_ids shape {token_ids.shape}"
  loss = cross_entropy(logits.view(-1, logits.size(-1)), token_ids.view(-1), reduction='none')
  perplexity = torch.exp(loss)
  return perplexity.view(token_ids.shape)


In [8]:
# for entry in tqdm(dataset.take(TOTAL_ENTRIES).to_iterable_dataset(), total=TOTAL_ENTRIES):
for entry in tqdm(dataset.select(range(TOTAL_ENTRIES))):
  chat = tokenizer.apply_chat_template(entry["conversation"], tokenize=False)
  inputs = tokenizer(chat, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model(**inputs)

  input_ids = inputs.input_ids[0]
  input_length = input_ids.size(0)
  target_ids = input_ids[1:]

  # Calculate perplexity from logits.
  # Remove the last token to avoid sampling the next token
  output_logits = outputs.logits[0, :-1, :]  # [batch_size, seq_len, vocab_size]
  full_perplexity = calc_perplexity(output_logits, target_ids)
  print(f"Full Perplexity: {full_perplexity.mean().item()}")

  base_top_k = model.config.num_experts_per_tok
  token_top_ks = torch.where(full_perplexity > 1.2, base_top_k, base_top_k-1)
  # Append the last token to match the input length
  token_top_ks = torch.cat([token_top_ks, torch.tensor([base_top_k], device=model.device)], dim=0)

  with torch.no_grad():
    outputs = model(
      **inputs,
      token_top_ks=token_top_ks,
    )

  output_logits = outputs.logits[0, :-1, :]
  reduced_perplexity = calc_perplexity(output_logits, target_ids)
  print(f"Reduced Perplexity: {reduced_perplexity.mean().item()}")

  0%|          | 0/1 [00:00<?, ?it/s]

Full Perplexity: 113816633344.0


100%|██████████| 1/1 [00:05<00:00,  5.87s/it]

Reduced Perplexity: 129385889792.0


In [9]:
ceval_data = load_dataset('ceval/ceval-exam', 'accountant', split='test')

In [12]:
from jinja2 import Template
doc_to_text_template = Template("{{question.strip()}}\nA. {{A}}\nB. {{B}}\nC. {{C}}\nD. {{D}}\n<think>")
chat = doc_to_text_template.render(ceval_data[0])
print(chat)

下列关于资本结构理论的说法中，不正确的是____。
A. 代理理论、权衡理论、有企业所得税条件下的MM理论，都认为企业价值与资本结构有关
B. 按照优序融资理论的观点，考虑信息不对称和逆向选择的影响，管理者偏好首选留存收益筹资，然后是发行新股筹资，最后是债务筹资
C. 权衡理论是对有企业所得税条件下的MM理论的扩展
D. 代理理论是对权衡理论的扩展
<think>


In [22]:
from transformers import StopStringCriteria, StoppingCriteriaList

inputs = tokenizer(chat, return_tensors="pt").to(model.device)
stop_util_end_think = StopStringCriteria(
  tokenizer=tokenizer,
  stop_strings=["</think>", "<|end_of_turn|>"],
)
outputs = model.generate(
  **inputs,
  max_new_tokens=4096,
  return_dict_in_generate=True,
  output_logits=True,
  stopping_criteria=StoppingCriteriaList([
    stop_util_end_think
  ]),
)

In [25]:
outputs.sequences[0].shape

torch.Size([3998])

In [23]:
temp_conversation = tokenizer.decode(outputs.sequences[0])

In [10]:
with open('data/ceval/temp_conversation.txt', 'r') as f:
  temp_conversation = f.read()

In [11]:
print(temp_conversation)

下列关于资本结构理论的说法中，不正确的是____。
A. 代理理论、权衡理论、有企业所得税条件下的MM理论，都认为企业价值与资本结构有关
B. 按照优序融资理论的观点，考虑信息不对称和逆向选择的影响，管理者偏好首选留存收益筹资，然后是发行新股筹资，最后是债务筹资
C. 权衡理论是对有企业所得税条件下的MM理论的扩展
D. 代理理论是对权衡理论的扩展
<think>
嗯，我现在要解决这个关于资本结构理论的选择题，题目是问哪个说法不正确。选项是A到D四个。首先，我需要回忆一下各个资本结构理论的基本内容，然后逐一分析每个选项是否正确。

首先，资本结构理论主要包括几个主要的理论：MM理论（包括有税和无税情况）、权衡理论、代理理论、优序融资理论等等。我需要先明确每个理论的核心观点，然后再看题目中的各个选项是否符合这些理论。

题目中的选项：

A选项说代理理论、权衡理论、有企业所得税条件下的MM理论都认为企业价值与资本结构有关。这个是否正确呢？首先，MM理论在有税的情况下，认为企业价值会随着负债的增加而增加，因为利息税盾的作用，所以企业价值与资本结构有关。权衡理论则是说企业会在债务的税盾利益和财务困境成本之间权衡，所以企业价值也与资本结构有关。代理理论的话，应该是指代理成本的存在，比如债务会导致股东和债权人之间的代理成本，或者股东与管理者之间的代理成本，这些都会影响企业价值，所以代理理论也应该认为资本结构影响企业价值。因此A选项的说法是对的，所以A正确，不是答案。

B选项说按照优序融资理论，考虑信息不对称和逆向选择的影响，管理者偏好首选留存收益筹资，然后是发行新股筹资，最后是债务筹资。这里可能有问题。根据优序融资理论，企业的融资顺序是首先内部融资（比如留存收益），然后是债务融资，最后才是发行新股。因为如果公司发行新股，可能被市场解读为股价被高估，导致逆向选择问题。所以正确的顺序应该是留存收益→债务→新股。而B选项中说“然后是发行新股筹资，最后是债务筹资”，这明显顺序错误，应该是先债务，再新股。所以B选项的说法不正确，可能就是答案。不过我需要再确认一下。

C选项说权衡理论是对有企业所得税条件下的MM理论的扩展。这里需要明确权衡理论和MM理论的关系。MM理论在有税的情况下，认为负债越多企业价值越高，但权衡理论则是在此基础上考虑了财务困境成本，所以权衡理论是对MM理论的扩展，

In [12]:
prefix_len = len(tokenizer.encode(chat))
prefix_len

256

In [ ]:
inputs = tokenizer(temp_conversation, return_tensors="pt").to(model.device)

with torch.no_grad():
  full_outputs = model(**inputs)

input_ids = inputs.input_ids[0]
input_length = input_ids.size(0)
target_ids = input_ids[1:]

# Calculate perplexity from logits.
# Remove the last token to avoid sampling the next token
full_output_logits = full_outputs.logits[0, :-1, :]  # [batch_size, seq_len, vocab_size]
full_perplexity = calc_perplexity(full_output_logits, target_ids)
print(f"Full Perplexity: {full_perplexity[prefix_len:].mean().item()}")

base_top_k = model.config.num_experts_per_tok
token_top_ks = base_top_k
token_top_ks = torch.where(full_perplexity < 2.0, base_top_k - 1, token_top_ks)
token_top_ks = torch.where(full_perplexity < 1.02, base_top_k - 2, token_top_ks)
token_top_ks = torch.where(full_perplexity < 1.004, base_top_k - 3, token_top_ks)
# token_top_ks = torch.where(full_perplexity < 1.1, base_top_k - 4, token_top_ks)
# token_top_ks = torch.where(full_perplexity > 4.0, base_top_k - 3, token_top_ks)
# Append the last token to match the input length
token_top_ks = torch.cat([token_top_ks, torch.tensor([base_top_k], device=model.device)], dim=0)

with torch.no_grad():
  reduced_outputs = model(
    **inputs,
    token_top_ks=token_top_ks,
  )

reduced_output_logits = reduced_outputs.logits[0, :-1, :]
reduced_perplexity = calc_perplexity(reduced_output_logits, target_ids)
print(f"Reduced Perplexity: {reduced_perplexity[prefix_len:].mean().item()}")

Full Perplexity: 1.546875
Reduced Perplexity: 9.9375


In [60]:
(full_perplexity < 1.004).sum() / full_perplexity.numel()

tensor(0.3544, device='cuda:0')

In [58]:
display_ppl_in_notebook(tokenizer.batch_decode(target_ids)[prefix_len:], full_perplexity.log().tolist()[prefix_len:])

In [59]:
display_ppl_in_notebook(tokenizer.batch_decode(target_ids)[prefix_len:], reduced_perplexity.log().tolist()[prefix_len:])

In [89]:
print(full_perplexity[prefix_len:].mean())
reduced_perplexity[prefix_len:].mean()

tensor(1.5312, device='cuda:0', dtype=torch.bfloat16)


tensor(1.6719, device='cuda:0', dtype=torch.bfloat16)

In [56]:
# show index and value of top-10
reduced_perplexity[prefix_len:].argsort(descending=True)[:10]

tensor([ 346,  368,  713, 3853, 2825, 2755,  839, 1283, 3587, 1091],
       device='cuda:0')

In [24]:
pos_to_inspect = 346
# pos_to_inspect = 368
# pos_to_inspect = 713

In [25]:
token_choices = full_output_logits[prefix_len:][pos_to_inspect]
top_tokens = token_choices.argsort(descending=True)[:10]
print(tokenizer.batch_decode(top_tokens))
print(token_choices[top_tokens])

['逆', '股价', '新股', '股东', '价值', '融资', '信息', '公司', '股权', '股票']
tensor([29.5000, 29.0000, 26.1250, 24.6250, 23.8750, 23.1250, 22.8750, 22.7500,
        22.7500, 22.6250], device='cuda:0', dtype=torch.bfloat16)


In [26]:
token_choices = reduced_output_logits[prefix_len:][pos_to_inspect]
top_tokens = token_choices.argsort(descending=True)[:10]
print(tokenizer.batch_decode(top_tokens))
print(token_choices[top_tokens])

['股价', '新股', '逆', '股东', '融资', '价值', '公司', '股权', '股票', '市值']
tensor([30.7500, 27.1250, 25.0000, 24.7500, 24.6250, 24.0000, 23.7500, 23.7500,
        23.1250, 22.8750], device='cuda:0', dtype=torch.bfloat16)


In [27]:
tokenizer.decode(target_ids[prefix_len:][pos_to_inspect])

'逆'

In [28]:
reduced_perplexity[prefix_len+pos_to_inspect]

tensor(324., device='cuda:0', dtype=torch.bfloat16)

In [31]:
(full_perplexity > 2.0).sum() / full_perplexity.shape[0]

tensor(0.1507, device='cuda:0')